In [ ]:
from graphviz import Digraph
import numpy as np
import pandas as pd
from datetime import datetime, timedelta


file_path = "bases/para tesis LCC_Datos.xlsx"
df_totales = pd.read_excel(file_path)

df_planes_por_alumno = pd.read_excel(file_path, sheet_name="planes por alumno")
df_cursadas_por_alumno = pd.read_excel(file_path, sheet_name="cursadas por alumno (2)")
df_lista_de_materias = pd.read_excel(file_path, sheet_name="lista de materias")
df_finales_por_alumno = pd.read_excel(file_path, sheet_name="finales por alumno") #sheet_name="finales por alumno")
df_estados_de_actas = pd.read_excel(file_path, sheet_name="estados de actas")
df_resultados_posibles = pd.read_excel(file_path, sheet_name="resultados posibles")

In [ ]:
year = 2023 #2021 para atras

In [ ]:
cursaronYEAR = df_cursadas_por_alumno[(df_cursadas_por_alumno['anio_academico'] == year) 
                                     & (df_cursadas_por_alumno['periodo'].isin(['1er Cuatrimestre', '1er Bimestre'])) 
                                     & (df_cursadas_por_alumno['codigo_materia'].isin(['COMP930001', 'COMP930007']))
                                     & (df_cursadas_por_alumno['persona'].isin(df_planes_por_alumno['persona']))  ][['persona']].drop_duplicates('persona').copy() #personaID que cursaron en YEAR
# print(len(cursaron2022))  

preYEAR = df_cursadas_por_alumno[df_cursadas_por_alumno['anio_academico'] < year][['persona']].drop_duplicates('persona') #personasID con inscripcion  pre-YEAR
camadaYEAR = cursaronYEAR.drop( cursaronYEAR[cursaronYEAR['persona'].isin( list(preYEAR.transpose().iloc[0].values) )].index ) #camada2022 = cursaron2022 SIN los que se han inscripto pre-YEAR
camadaYEAR

In [ ]:
dic_materias = df_lista_de_materias[(df_lista_de_materias['codigo_materia'].str.contains('LCDA21')) | (df_lista_de_materias['codigo_materia'].isin(["COMP930003","COMP930004","COMP930005", "COMP930001","COMP230008", "COMP230001", "COMP230015"]))].sort_values(by="codigo_materia")[:17].drop_duplicates().reset_index().drop(columns=('index'))

dic_materias = dic_materias.sort_values('codigo_materia', ascending= False).reset_index()
dic_materias['index'] = list(range(13)) + [8,10,12,11]
dic_materias

In [ ]:
#todos los finales de los alumnos de la 
# - camada de year, 
# - finales aprobados,
# - y es de las materias obligatorias

finales_camadaYEAR = df_finales_por_alumno[(df_finales_por_alumno['resultado'] == 'A') & 
                                           (df_finales_por_alumno['persona'].isin(camadaYEAR['persona'].to_list())) & 
                                           (df_finales_por_alumno['codigo_materia'].isin(dic_materias['codigo_materia'].tolist())) ]
#finales_camadaYEAR

In [ ]:
# finales_camadaYEAR[ finales_camadaYEAR['persona'] =  ]

In [ ]:
#agrego vector con todos ceros
finales_camadaYEAR.insert(len(finales_camadaYEAR.columns), 'vector', [[0]*(dic_materias['index'].max()+1) for i in range(len(finales_camadaYEAR)) ]  )
finales_camadaYEAR = finales_camadaYEAR.sort_values(['persona', 'fecha_cierre_acta' ] ) # hago sort por persona, luego year, luego periodo
finales_camadaYEAR

##### recorrer a los finales en ese orden

In [ ]:
for filaNro in range(1, len(finales_camadaYEAR)):
    oldFila = finales_camadaYEAR.iloc[filaNro-1]
    fila = finales_camadaYEAR.iloc[filaNro]

    if int(oldFila.persona) == int(fila.persona):
        finales_camadaYEAR.at[int(fila.name), 'vector']  = oldFila.vector

    if fila.resultado == 'A':
        newVect = finales_camadaYEAR.loc[int(fila.name), 'vector'].copy()
        k = int(dic_materias[dic_materias['codigo_materia'] == fila.codigo_materia ]['index'])
        newVect[k] = 1
        finales_camadaYEAR.at[int(fila.name), 'vector'] = newVect

finales_camadaYEAR

In [ ]:
finales_camadaYEAR['vectStr'] = finales_camadaYEAR['vector'].apply(str) # agrego copia de fila 'vector' pero como string
# finales_camadaYEAR

#### Segunda pasada

In [ ]:
finales_camadaYEAR[['vectStr', 'vector']].copy().drop_duplicates('vectStr').reset_index().drop(columns=('index')) 

In [ ]:
vectorCeroStandar = str([0]*(dic_materias['index'].max()+1))
edges = []
persona = []
esLoop = []

#asocia nodo <-> numero
dic_node = finales_camadaYEAR[['vectStr', 'vector']].copy().drop_duplicates('vectStr').reset_index().drop(columns=('index')) # index es el nombre del nodo que usaré para nodos de grafic
if(len(dic_node[dic_node['vectStr'] == vectorCeroStandar]) == 0):
    dic_node.loc[-1] = [vectorCeroStandar, [0]*(dic_materias['index'].max()+1) ]
    dic_node.index = dic_node.index + 1

for p in finales_camadaYEAR['persona'].drop_duplicates().tolist():
    personaCursadas = finales_camadaYEAR[finales_camadaYEAR['persona'] == p]

    vectA = vectorCeroStandar

    oldFila = personaCursadas.iloc[0]
    oldDate = oldFila.fecha_cierre_acta

    
    for i in range(0,len(personaCursadas)): 
        fila = personaCursadas.iloc[i]
        laDate = fila.fecha_cierre_acta

        if oldDate!= laDate: 
            originPoint = int(dic_node[dic_node['vectStr'] == vectA].iloc[0].name) 
            endPoint = int(dic_node[dic_node['vectStr'] == oldFila.vectStr ].iloc[0].name)

            edges.append((originPoint, endPoint))
            persona.append(int(fila.persona))
            esLoop.append(originPoint==endPoint)

            vectA = oldFila.vectStr


        oldFila = fila
        oldDate = laDate

        isAproved = fila.resultado


    if vectA!=oldFila.vectStr and isAproved=='A':
        originPoint = int(dic_node[dic_node['vectStr'] == vectA].iloc[0].name) 
        endPoint = int(dic_node[dic_node['vectStr'] == oldFila.vectStr ].iloc[0].name)
    
        edges.append((originPoint, endPoint))
        persona.append(int(fila.persona))
        esLoop.append(originPoint==endPoint)
    
        
# edges

In [ ]:
allEdges = pd.DataFrame({'edges' : edges, 'persona': persona, 'esLoop': esLoop})

#Loops 
losLoops = allEdges[allEdges['esLoop']].drop_duplicates().copy()
losEdges = allEdges[~(allEdges['esLoop'])]

allAllEdges = pd.concat([losEdges,losLoops])


df_edges = pd.DataFrame({'edge': allAllEdges['edges'].drop_duplicates()  })
df_edges.insert(1, 'appears', [ len(allAllEdges[allAllEdges['edges'] == df_edges['edge'].iloc[i]]) for i in range(len(df_edges)) ] ) 
df_edges

### Hacer grafico

In [ ]:
materiasCatalogo = pd.merge(dic_materias, df_lista_de_materias, on='codigo_materia', how='left')
materiasCatalogo['codigo'] = ['Intr.MC','Intr.IOO','Intr.ECdD','Proba','ALC', 'Avanzado', 'LaboDatos','Analisis2', 'Algebra','Analisis1','Algo3','Algo2','Algo1','Algebra','Algo3','Algo1', 'Algo2']
materiasCatalogo = materiasCatalogo[:13]

def strMateriasAprobadas(l): #l <- nodosCaminos.iloc[i]['vector']
    boolean_mask = [bool(valor) for valor in l]
    # temp= pd.merge(dic_materias[boolean_mask], df_lista_de_materias, on='codigo_materia', how='left')
    output = f""
    for i in range(len(materiasCatalogo[boolean_mask])): #  materiasCatalogo[boolean_mask]['codigo'].tolist():
        output+= str(materiasCatalogo[boolean_mask].iloc[i].name+1) +'. '+ materiasCatalogo[boolean_mask].iloc[i].codigo +'\n'
    return output[:len(output)-1]


##### sugerencia de Mati: limitar a nodos con aristas de >= 3 personas

In [ ]:
#df_edges = df_edges[df_edges['appears'] >= 2 ]

#Plan del 2022
nodosDelPlan = [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 
                [0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0], #algebra y analisis
                [0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1], #algo1, algebra y analisis
                [0, 0, 0, 0, 1, 0, 1, 1, 1, 1, 0, 0, 1], #analisis2, alc, labodatos algo1, algebra y analisis
                [0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1], #algo2, Avanzado, analisis2, alc, labodatos algo1, algebra y analisis
                [0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], #algo3, proba algo2, Avanzado, analisis2, alc, labodatos algo1, algebra y analisis
                [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], #algo3, proba algo2, Avanzado, analisis2, alc, labodatos algo1, algebra y analisis
                ]

In [ ]:
tpl = set()
for tuple in df_edges['edge'].tolist():
    tpl.add(tuple[0])
    tpl.add(tuple[1])
#tpl son los index de los nodos que SI les llega o sale una edge de ellos
tpl = list(tpl)

In [ ]:
dot = Digraph()
for i in tpl: #range(len(dic_node)):
    nombre =  strMateriasAprobadas(dic_node.loc[i]['vector'])

    if(dic_node.loc[i]['vector'] in nodosDelPlan):
        elColor="#62b3ff"
    else:
        elColor="#000000"
    
    dot.node(str(i), str(i) +'\n' + nombre, color = elColor)

for i in range(len(df_edges)):
    beginning = df_edges['edge'].iloc[i][0]
    end = df_edges['edge'].iloc[i][1]

    width = df_edges['appears'].iloc[i]

    if((dic_node.loc[beginning]['vector'] in nodosDelPlan) and 
       (dic_node.loc[end]['vector'] in nodosDelPlan) and 
       abs(nodosDelPlan.index(dic_node.loc[beginning]['vector']) - nodosDelPlan.index(dic_node.loc[end]['vector'])) == 1):
        elColor="#62b3ff"
    else:
        elColor="#5c5c5c"
    dot.edge(str(beginning), str(end), penwidth = str( width/4 ), label=str( width ), color=elColor)

dot

___________________________________
### Distancia con Practico y Final  
fecha_init_cuatri_aprox: es una medida aproximada de la fecha INICIAL de cursada. Duda: donde estan los ANEXOS de las "Ordenes del dia"?

In [ ]:
### Elijo personas de las materias obligatorias, y aprobadas   comment: ese YEAR camada, en
cursadas = df_cursadas_por_alumno[#df_cursadas_por_alumno['persona'].isin(camadaYEAR['persona'].tolist()) & 
                       (df_cursadas_por_alumno['codigo_materia'].isin(dic_materias['codigo_materia'].tolist())) &
                       (df_cursadas_por_alumno['resultado'] == 'A')]

cursadas['fecha_init_cuatri_aprox'] = [ datetime.strptime(str(cursadas.loc[i].anio_academico)+"-"+('03' if cursadas.loc[i].periodo == '1er Cuatrimestre' else ( '01' if cursadas.loc[i].periodo == '1er Bimestre' else '08'))+('-20' if cursadas.loc[i].periodo == '1er Bimestre' else "-15"), "%Y-%m-%d") for i in cursadas.index]
cursadas

In [ ]:
finales = df_finales_por_alumno[#df_finales_por_alumno['persona'].isin(camadaYEAR['persona'].tolist()) & 
                      (df_finales_por_alumno['codigo_materia'].isin(dic_materias['codigo_materia'].tolist())) & 
                      (df_finales_por_alumno['resultado'] == 'A') &
                      (df_finales_por_alumno['instancia'] == 3)]
finales

In [ ]:
cursadas_y_finales = pd.merge(cursadas, finales, on=['persona', 'codigo_materia'], how='left')
cursadas_y_finales.sort_values(['persona', 'anio_academico', 'periodo' ])

In [ ]:
"Aprox cantidad de dias en un cuatrimestre: " + str(timedelta(days=7*16))[:8]

#### CalidadDeDatos: fecha_cierre_acta este bien (proxima al cuatri cuando se impartió la materia)

In [ ]:
cursadas_y_finales['dist_cierrerActa_y_finCuatriAprox'] = [(cursadas_y_finales.iloc[i].fecha_cierre_acta_x - (cursadas_y_finales.iloc[i].fecha_init_cuatri_aprox + (timedelta(days=7*7) if cursadas_y_finales.iloc[i].periodo == '1er Bimestre' else timedelta(days=7*16) ))   ) for i in cursadas_y_finales.index]
cursadas_y_finales[['persona', 'codigo_materia', 'periodo', 'anio_academico', 'fecha_init_cuatri_aprox', 'fecha_cierre_acta_x',  'dist_cierrerActa_y_finCuatriAprox']]

In [ ]:
avg_dist_cierrerActa_y_finCuatriAprox = cursadas_y_finales.groupby(['codigo_materia'])
df_calidadDatosActa = pd.merge(dic_materias, avg_dist_cierrerActa_y_finCuatriAprox['dist_cierrerActa_y_finCuatriAprox'].mean(), on='codigo_materia', how='left')
df_calidadDatosActa

#### para AVG distancia TP vs Final

In [ ]:
cursadas_y_finales['dist_tp_al_final'] = [(cursadas_y_finales.iloc[i].fecha_llamado_mesa - cursadas_y_finales.iloc[i].fecha_init_cuatri_aprox  ) for i in cursadas_y_finales.index] #- timedelta(days=7*16)
cursadas_y_finales[['persona', 'codigo_materia', 'periodo', 'anio_academico', 'fecha_init_cuatri_aprox','fecha_cierre_acta_x', 'fecha_llamado_mesa', 'dist_tp_al_final' ]]

### Ranking NO colgar el final por materia

In [ ]:
avg_dist_tp_al_final = cursadas_y_finales.groupby('codigo_materia')
df_calidadDatos = pd.merge(dic_materias, avg_dist_tp_al_final['dist_tp_al_final'].mean(), on='codigo_materia', how='left')
df_calidadDatos

Si le resto el average de una cursada:

In [ ]:
temp = [(df_calidadDatos.iloc[i].dist_tp_al_final - timedelta(days=7*16)) for i in df_calidadDatos.index]
df_calidadDatos['dist_tp_al_final'] = temp
df_calidadDatos.sort_values('dist_tp_al_final')